Cell 1 — Setup and automatic path detection
This cell searches your attached Kaggle datasets instead of assuming that every folder has the same path.

In [1]:

# =============================================================================
# CELL 1: Setup — paths, GPU, model, shared utilities
# =============================================================================

import os
import sys
import json
import time
import pickle
import subprocess
from pathlib import Path

print("Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics", "ensemble-boxes", "pycocotools"],
    check=True
)

import numpy as np
import pandas as pd
import cv2
import torch
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# ------------------------------------------------------------------
# 1. Find files in attached Kaggle datasets
# ------------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "wbf_optimization"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nSearching for common.py...")
common_candidates = list(INPUT_ROOT.rglob("common.py"))

if not common_candidates:
    raise FileNotFoundError(
        "common.py not found. Make sure your my_dataset dataset is attached."
    )

# Prefer common.py inside a folder named my_dataset
preferred = [p for p in common_candidates if p.parent.name == "my_dataset"]
COMMON_PATH = preferred[0] if preferred else common_candidates[0]

sys.path.insert(0, str(COMMON_PATH.parent))
print("Using common.py:", COMMON_PATH)

from common import *

print("\nSearching for competition data...")
csv_candidates = list(INPUT_ROOT.rglob("test_images.csv"))

valid_comp = [
    p.parent for p in csv_candidates
    if (p.parent / "val" / "images").exists()
    and (p.parent / "val" / "labels").exists()
]

if len(valid_comp) == 0:
    raise FileNotFoundError(
        "Could not find competition root containing test_images.csv "
        "and val/images + val/labels."
    )

if len(valid_comp) > 1:
    print("Found multiple competition folders:")
    for p in valid_comp:
        print(" -", p)
    raise RuntimeError("Please select the correct COMP_DIR manually.")

COMP_DIR = valid_comp[0]
VAL_DIR = COMP_DIR / "val"

print("Competition data:", COMP_DIR)

print("\nSearching for YOLO checkpoint...")
checkpoint_candidates = list(INPUT_ROOT.rglob("best.pt"))

preferred_checkpoints = [
    p for p in checkpoint_candidates
    if "yolov8l_800" in str(p)
]

if len(preferred_checkpoints) == 1:
    BEST_PT = preferred_checkpoints[0]
elif len(checkpoint_candidates) == 1:
    BEST_PT = checkpoint_candidates[0]
else:
    print("Available best.pt files:")
    for p in checkpoint_candidates:
        print(" -", p)
    raise RuntimeError(
        "Could not uniquely identify YOLO checkpoint. "
        "Set BEST_PT manually in this cell."
    )

print("YOLO checkpoint:", BEST_PT)

# ------------------------------------------------------------------
# 2. GPU
# ------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = 0
    print("\nGPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("\nWARNING: GPU not available.")

# ------------------------------------------------------------------
# 3. Experiment settings
# ------------------------------------------------------------------

IMGSZ = 800
YOLO_CONF = 0.20
YOLO_NMS_IOU = 0.45
MAX_DET = 1000

PRED_CACHE = OUT_DIR / "val_tta_predictions.pkl"
GT_JSON = OUT_DIR / "val_ground_truth.json"

print("\n" + "=" * 65)
print("SETUP COMPLETE")
print("=" * 65)
print("Model:", BEST_PT)
print("Validation:", VAL_DIR)
print("Image size:", IMGSZ)
print("YOLO confidence:", YOLO_CONF)
print("YOLO NMS IoU:", YOLO_NMS_IOU)
print("Output:", OUT_DIR)


Installing dependencies...
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings

Searching for common.py...
Using common.py: /kaggle/input/datasets/suchok/my-dataset/common.py

Searching for competition data...
Competition data: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026

Searching for YOLO checkpoint...
YOLO checkpoint: /kaggle/input/datasets/suchok/yolo-outputs/runs/detect/visdrone_yolo/yolov8l_800/weights/best.pt

GPU: Tesla T4

SETUP COMPLETE
Model: /kaggle/input/datasets/suchok/yolo-outputs/runs/detect/visdrone_yolo/yolov8l_800/weights/best.pt
Validation: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/val
Image size: 800
YOLO confidence: 0.2
YOLO NMS IoU: 0.45
Output: /kaggle/working/w

Cell 2 — Prepare the full validation set
We want to use all available validation frames, not the old 300-frame subset.

In [2]:

# =============================================================================
# CELL 2: Build full validation index and ground truth
# =============================================================================

print("Indexing validation images...")

val_index = index_split(VAL_DIR)

print("Validation frames:", len(val_index))
print("Columns:", list(val_index.columns))

assert len(val_index) > 0, "No validation images found."

# Check that image and label paths exist
missing_images = sum(
    not Path(p).exists() for p in val_index["image_path"]
)
missing_labels = sum(
    not Path(p).exists() for p in val_index["label_path"]
)

print("Missing images:", missing_images)
print("Missing labels:", missing_labels)

assert missing_images == 0, "Some validation images are missing."

# Build COCO ground truth, including ignore regions when available
if not GT_JSON.exists():
    print("\nBuilding COCO ground truth...")
    build_coco_gt(
        val_index,
        str(GT_JSON),
        include_ignore=True
    )
else:
    print("\nUsing cached ground truth:", GT_JSON)

print("\nValidation preparation complete.")
print("Ground truth:", GT_JSON)


Indexing validation images...
Validation frames: 2846
Columns: ['clip', 'frame', 'image_path', 'label_path', 'ignore_path']
Missing images: 0
Missing labels: 0

Building COCO ground truth...

Validation preparation complete.
Ground truth: /kaggle/working/wbf_optimization/val_ground_truth.json


Cell 3 — Run YOLO TTA once and cache predictions
This is the expensive cell. It runs the original and flipped image through YOLO, using the same settings as your current winning submission.

The predictions are saved so that all later WBF experiments can reuse them.

In [3]:

# =============================================================================
# CELL 3: Generate and cache original + horizontal flip predictions
# =============================================================================

if PRED_CACHE.exists():
    print("Prediction cache already exists:")
    print(PRED_CACHE)
    print("To regenerate, delete PRED_CACHE and rerun this cell.")

else:
    print("Loading YOLO...")
    model = YOLO(str(BEST_PT))

    all_predictions = []
    start_time = time.time()

    for i, (_, row) in enumerate(val_index.iterrows(), start=1):

        img = cv2.imread(row["image_path"])

        if img is None:
            raise RuntimeError(f"Could not read {row['image_path']}")

        h, w = img.shape[:2]

        # Original image
        result_orig = model.predict(
            source=img,
            imgsz=IMGSZ,
            conf=YOLO_CONF,
            iou=YOLO_NMS_IOU,
            max_det=MAX_DET,
            device=DEVICE,
            verbose=False
        )[0]

        # Horizontally flipped image
        img_flip = cv2.flip(img, 1)

        result_flip = model.predict(
            source=img_flip,
            imgsz=IMGSZ,
            conf=YOLO_CONF,
            iou=YOLO_NMS_IOU,
            max_det=MAX_DET,
            device=DEVICE,
            verbose=False
        )[0]

        def extract_predictions(result, flipped=False):
            if result.boxes is None or len(result.boxes) == 0:
                return (
                    np.zeros((0, 4), dtype=np.float32),
                    np.zeros((0,), dtype=np.float32),
                    np.zeros((0,), dtype=np.float32)
                )

            boxes = result.boxes.xyxy.cpu().numpy().astype(np.float32)
            scores = result.boxes.conf.cpu().numpy().astype(np.float32)
            labels = result.boxes.cls.cpu().numpy().astype(np.float32)

            if flipped:
                x1 = boxes[:, 0].copy()
                x2 = boxes[:, 2].copy()
                boxes[:, 0] = w - x2
                boxes[:, 2] = w - x1

            return boxes, scores, labels

        orig = extract_predictions(result_orig, flipped=False)
        flip = extract_predictions(result_flip, flipped=True)

        all_predictions.append({
            "width": w,
            "height": h,
            "orig": orig,
            "flip": flip
        })

        if i % 250 == 0 or i == len(val_index):
            elapsed = (time.time() - start_time) / 60
            print(
                f"[{i}/{len(val_index)}] "
                f"elapsed={elapsed:.1f} min"
            )

    with open(PRED_CACHE, "wb") as f:
        pickle.dump(all_predictions, f)

    print("\nSaved predictions:", PRED_CACHE)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Prediction cache ready.")


Loading YOLO...
[250/2846] elapsed=0.7 min
[500/2846] elapsed=1.4 min
[750/2846] elapsed=2.1 min
[1000/2846] elapsed=2.8 min
[1250/2846] elapsed=3.5 min
[1500/2846] elapsed=4.3 min
[1750/2846] elapsed=5.1 min
[2000/2846] elapsed=6.0 min
[2250/2846] elapsed=6.9 min
[2500/2846] elapsed=7.6 min
[2750/2846] elapsed=8.3 min
[2846/2846] elapsed=8.6 min

Saved predictions: /kaggle/working/wbf_optimization/val_tta_predictions.pkl
Prediction cache ready.


Cell 4 — Run the WBF experiment grid
This is where we compare the configurations. We keep YOLO's NMS IoU fixed at 0.45, so we can isolate the effect of WBF.

In [4]:

# =============================================================================
# CELL 4: WBF configuration comparison on full validation set
# =============================================================================

with open(PRED_CACHE, "rb") as f:
    all_predictions = pickle.load(f)

assert len(all_predictions) == len(val_index)

EXPERIMENTS = [
    {"name": "baseline_avg_055", "iou": 0.55, "conf_type": "avg"},
    {"name": "avg_045",          "iou": 0.45, "conf_type": "avg"},
    {"name": "avg_050",          "iou": 0.50, "conf_type": "avg"},
    {"name": "avg_060",          "iou": 0.60, "conf_type": "avg"},
    {"name": "max_055",          "iou": 0.55, "conf_type": "max"},
]

results = []

for exp in EXPERIMENTS:

    print("\n" + "=" * 65)
    print("Running:", exp["name"])
    print("=" * 65)

    coco_predictions = []

    for image_idx, pred in enumerate(all_predictions, start=1):

        img_w = pred["width"]
        img_h = pred["height"]

        boxes_list = []
        scores_list = []
        labels_list = []

        for boxes, scores, labels in [pred["orig"], pred["flip"]]:

            boxes = boxes.copy()

            if len(boxes) > 0:
                boxes[:, [0, 2]] /= img_w
                boxes[:, [1, 3]] /= img_h
                boxes = np.clip(boxes, 0, 1)

            boxes_list.append(boxes)
            scores_list.append(scores.copy())
            labels_list.append(labels.copy())

        if not any(len(b) > 0 for b in boxes_list):
            continue

        boxes_fused, scores_fused, labels_fused = weighted_boxes_fusion(
            boxes_list,
            scores_list,
            labels_list,
            weights=[1, 1],
            iou_thr=exp["iou"],
            skip_box_thr=0.001,
            conf_type=exp["conf_type"],
            allows_overflow=False
        )

        # Convert normalized boxes back to original image pixels
        boxes_fused[:, [0, 2]] *= img_w
        boxes_fused[:, [1, 3]] *= img_h

        # Keep the top 500 predictions per image
        order = np.argsort(-scores_fused)[:500]

        for idx in order:
            x1, y1, x2, y2 = boxes_fused[idx]

            bw = float(x2 - x1)
            bh = float(y2 - y1)

            if bw <= 0 or bh <= 0:
                continue

            cls_id = int(labels_fused[idx])

            if cls_id < 0 or cls_id >= NUM_CLASSES:
                continue

            coco_predictions.append({
                "image_id": image_idx,
                "category_id": cls_id + 1,
                "bbox": [
                    float(x1),
                    float(y1),
                    bw,
                    bh
                ],
                "score": float(scores_fused[idx])
            })

    pred_json = OUT_DIR / f"{exp['name']}_predictions.json"

    with open(pred_json, "w") as f:
        json.dump(coco_predictions, f)

    metrics = coco_eval_summary(
        str(GT_JSON),
        str(pred_json)
    )

    row = {
        "experiment": exp["name"],
        "wbf_iou": exp["iou"],
        "conf_type": exp["conf_type"],
        "detections": len(coco_predictions),
        **metrics
    }

    results.append(row)

    print("Detections:", len(coco_predictions))
    print("mAP:", round(metrics["mAP"], 5))
    print("AP50:", round(metrics["AP50"], 5))
    print("AP75:", round(metrics["AP75"], 5))
    print("AP small:", round(metrics["AP_small"], 5))

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv(OUT_DIR / "wbf_experiment_results.csv", index=False)

print("\n" + "=" * 75)
print("WBF EXPERIMENT RESULTS")
print("=" * 75)

display(
    results_df.sort_values("mAP", ascending=False).reset_index(drop=True)
)

print("\nSaved:", OUT_DIR / "wbf_experiment_results.csv")



Running: baseline_avg_055
Detections: 133671
mAP: 0.19105
AP50: 0.36469
AP75: 0.17528
AP small: 0.09391

Running: avg_045
Detections: 132328
mAP: 0.19079
AP50: 0.36477
AP75: 0.17521
AP small: 0.0938

Running: avg_050
Detections: 133063
mAP: 0.19095
AP50: 0.36474
AP75: 0.17527
AP small: 0.09386

Running: avg_060
Detections: 134442
mAP: 0.19118
AP50: 0.36451
AP75: 0.17536
AP small: 0.0939

Running: max_055
Detections: 133671
mAP: 0.18763
AP50: 0.35754
AP75: 0.17198
AP small: 0.09287

WBF EXPERIMENT RESULTS


,experiment,wbf_iou,conf_type,detections,mAP,AP50,AP75,AP_small,AP_medium,AP_large
0,avg_060,0.60,avg,134442,0.191175,0.364514,0.175357,0.093905,0.179525,0.333882
1,baseline_avg_055,0.55,avg,133671,0.191045,0.364690,0.175278,0.093915,0.179496,0.333910
2,avg_050,0.50,avg,133063,0.190955,0.364742,0.175266,0.093864,0.179470,0.333876
3,avg_045,0.45,avg,132328,0.190789,0.364768,0.175205,0.093801,0.179174,0.333166
4,max_055,0.55,max,133671,0.187634,0.357541,0.171976,0.092874,0.176492,0.326762



Saved: /kaggle/working/wbf_optimization/wbf_experiment_results.csv


Cell 5 — Select the best configuration
This cell identifies the best local validation configuration and compares it with your baseline.


# =============================================================================
# CELL 5: Results analysis
# =============================================================================

results_df = pd.read_csv(OUT_DIR / "wbf_experiment_results.csv")

baseline = results_df[
    results_df["experiment"] == "baseline_avg_055"
].iloc[0]

best = results_df.sort_values("mAP", ascending=False).iloc[0]

print("=" * 65)
print("BEST WBF CONFIGURATION")
print("=" * 65)

print("Baseline:", baseline["experiment"])
print("Baseline mAP:", round(baseline["mAP"], 5))
print("Baseline AP50:", round(baseline["AP50"], 5))

print("\nBest:", best["experiment"])
print("Best WBF IoU:", best["wbf_iou"])
print("Best confidence type:", best["conf_type"])
print("Best mAP:", round(best["mAP"], 5))
print("Best AP50:", round(best["AP50"], 5))

print("\nImprovement over baseline:")
print("mAP:", round(best["mAP"] - baseline["mAP"], 5))
print("AP50:", round(best["AP50"] - baseline["AP50"], 5))

if best["experiment"] == baseline["experiment"]:
    print("\nNo tested WBF configuration improved the baseline.")
else:
    print("\nA different WBF configuration performed better locally.")

print("\nIMPORTANT:")
print("These are local validation results, not Kaggle leaderboard scores.")
print("Do not replace your 0.26189 submission until a new submission")
print("has been tested on Kaggle.")
